In [ ]:
%%bash
rm -rf /tmp/dbt-fabric-bundle
tar -xzf /lakehouse/default/Files/dbt-fabric-bundle.tar.gz -C /tmp
pip install -q --no-index --find-links=/tmp/dbt-fabric-bundle/wheels dbt-core dbt-fabricspark

In [ ]:
import os
PROJECT, TARGET = "dbt-adventureworks", "fabric-fabric"
PROJECT_DIR = f"/tmp/dbt-fabric-bundle/projects/{PROJECT}"
os.environ["DBT_PROFILES_DIR"] = PROJECT_DIR

from dbt.cli.main import dbtRunner
for cmd in [["deps"], ["run"], ["test"]]:
    dbtRunner().invoke(cmd + ["--project-dir", PROJECT_DIR, "--target", TARGET])

In [ ]:
import requests, notebookutils, yaml

cfg = yaml.safe_load(open(f"{PROJECT_DIR}/profiles.yml"))["adventureworks"]["outputs"][TARGET]
session_id = open(cfg["session_id_file"]).read().strip()

url = f"https://api.fabric.microsoft.com/v1/workspaces/{cfg['workspaceid']}/lakehouses/{cfg['lakehouseid']}/livyApi/versions/2023-12-01/sessions/{session_id}"
r = requests.delete(url, headers={"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"})
print(f"Delete session {session_id}: {r.status_code} {r.reason}")